# 03 - Inference and Submission for Akkadian-to-English Translation

This notebook generates the final submission for the Deep Past Challenge.

**Key features:**
- Enhanced post-processing for chrF++ optimization
- Proper noun handling using onomasticon dictionary
- Length-adaptive beam search
- Optimized inference with mixed precision

In [ ]:
import os
import gc
import re
import json
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Environment setup for performance
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
@dataclass
class InferenceConfig:
    # Paths - Update these for Kaggle
    test_data_path: str = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    model_path: str = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"  # Baseline model
    # model_path: str = "./akkadian-byt5-finetuned/final"  # Or your fine-tuned model
    proper_noun_dict_path: str = "proper_noun_dict.json"
    output_dir: str = "/kaggle/working"
    
    # Generation parameters - OPTIMIZED for T4 GPUs
    max_input_length: int = 512
    max_new_tokens: int = 512
    num_beams: int = 8
    length_penalty: float = 1.5
    early_stopping: bool = True
    no_repeat_ngram_size: int = 0
    
    # Batch processing - REDUCED for T4 memory
    batch_size: int = 4  # Reduced from 8 for T4
    num_workers: int = 2
    
    # Optimization
    use_mixed_precision: bool = True
    use_better_transformer: bool = True
    use_length_adaptive: bool = True
    
    # Post-processing
    use_proper_noun_correction: bool = True
    aggressive_postprocessing: bool = True
    
    # Task prefix
    task_prefix: str = "translate Akkadian to English: "
    
    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)
        if not torch.cuda.is_available():
            self.use_mixed_precision = False
            self.use_better_transformer = False

config = InferenceConfig()
print(f"Device: {config.device}")
print(f"Model: {config.model_path}")
print(f"Batch size: {config.batch_size}")

## 1. Preprocessing Functions

In [ ]:
class AkkadianPreprocessor:
    """
    Preprocess Akkadian transliteration text.
    """
    def __init__(self):
        self.patterns = {
            'big_gap': re.compile(r'(\.{3,}|…+|……|\[\s*\.+\s*\])'),
            'small_gap': re.compile(r'(\[x+\]|xx+|\s+x\s+)'),
            'uncertainty': re.compile(r'[!?]'),
            'brackets': re.compile(r'\[([^\]]+)\]'),
            'half_brackets': re.compile(r'[˹⌈]([^˺⌉]+)[˺⌉]'),
            'whitespace': re.compile(r'\s+')
        }
    
    def preprocess(self, text: str) -> str:
        if pd.isna(text) or not str(text).strip():
            return ""
        
        text = str(text)
        
        # Normalize gaps
        text = self.patterns['big_gap'].sub('<big_gap>', text)
        text = self.patterns['small_gap'].sub('<gap>', text)
        
        # Remove uncertainty markers
        text = self.patterns['uncertainty'].sub('', text)
        
        # Remove brackets but keep content
        text = self.patterns['brackets'].sub(r'\1', text)
        text = self.patterns['half_brackets'].sub(r'\1', text)
        
        # Normalize whitespace
        text = self.patterns['whitespace'].sub(' ', text).strip()
        
        return text
    
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        return [self.preprocess(t) for t in texts]

preprocessor = AkkadianPreprocessor()
print("Preprocessor initialized")

## 2. Post-Processing Functions

In [ ]:
class EnhancedPostprocessor:
    """
    Enhanced post-processing for chrF++ optimization.
    """
    def __init__(self, proper_noun_dict: Optional[Dict[str, str]] = None, aggressive: bool = True):
        self.proper_noun_dict = proper_noun_dict or {}
        self.aggressive = aggressive
        
        # Compiled patterns
        self.patterns = {
            'gap': re.compile(r'(\[x\]|\(x\)|\bx\b)', re.I),
            'big_gap': re.compile(r'(\.{3,}|…|\[\.+\])'),
            'annotations': re.compile(r'\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)', re.I),
            'repeated_words': re.compile(r'\b(\w+)(?:\s+\1\b)+'),
            'whitespace': re.compile(r'\s+'),
            'punct_space': re.compile(r'\s+([.,:;!?])'),
            'repeated_punct': re.compile(r'([.,;:])\1+'),
            'quotes': re.compile(r'"{2,}'),
        }
        
        # Character transformations
        self.subscript_trans = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')
        self.special_chars_trans = str.maketrans('ḫḪ', 'hH')
        self.forbidden_chars = '——<>⌈⌋⌊[]+ʾ/;'
        self.forbidden_trans = str.maketrans('', '', self.forbidden_chars)
        
        # Fraction mappings
        self.fractions = {
            r'0\.5\b': '½',
            r'0\.25\b': '¼',
            r'0\.75\b': '¾',
            r'0\.33333\b': '1/3',
            r'0\.66666\b': '2/3',
            r'0\.83333\b': '5/6',
            r'0\.16666\b': '1/6',
        }
        
        # Build proper noun patterns for efficient matching
        self._build_proper_noun_patterns()
    
    def _build_proper_noun_patterns(self):
        """Build sorted proper noun list for efficient matching."""
        # Sort by length (longest first) to avoid partial matches
        self.sorted_proper_nouns = sorted(
            self.proper_noun_dict.items(), 
            key=lambda x: len(x[0]), 
            reverse=True
        )
    
    def _fix_proper_nouns(self, text: str) -> str:
        """Apply proper noun corrections."""
        if not self.proper_noun_dict:
            return text
        
        # Apply corrections for longer matches first
        for variant, canonical in self.sorted_proper_nouns[:1000]:  # Limit for performance
            if variant in text.lower():
                # Case-insensitive replacement preserving some case
                pattern = re.compile(re.escape(variant), re.IGNORECASE)
                text = pattern.sub(canonical, text)
        
        return text
    
    def _normalize_fractions(self, text: str) -> str:
        """Convert decimal fractions to standard forms."""
        for pattern, replacement in self.fractions.items():
            text = re.sub(pattern, replacement, text)
        
        # Handle fractions with preceding numbers: "2.5" -> "2½"
        text = re.sub(r'(\d+)\.5\b', r'\g<1>½', text)
        text = re.sub(r'(\d+)\.25\b', r'\g<1>¼', text)
        text = re.sub(r'(\d+)\.75\b', r'\g<1>¾', text)
        
        return text
    
    def _remove_repeated_phrases(self, text: str) -> str:
        """Remove repeated words and multi-word phrases."""
        # Single word repetitions
        text = self.patterns['repeated_words'].sub(r'\1', text)
        
        # Multi-word phrase repetitions (2-4 words)
        for n in range(4, 1, -1):
            pattern = r'\b((?:\w+\s+){' + str(n - 1) + r'}\w+)(?:\s+\1\b)+'
            text = re.sub(pattern, r'\1', text)
        
        return text
    
    def postprocess(self, text: str) -> str:
        """Apply all post-processing steps."""
        if not isinstance(text, str) or not text.strip():
            return ""
        
        # Basic character transformations
        text = text.translate(self.special_chars_trans)
        text = text.translate(self.subscript_trans)
        
        if self.aggressive:
            # Gap normalization
            text = self.patterns['gap'].sub('<gap>', text)
            text = self.patterns['big_gap'].sub('<big_gap>', text)
            text = text.replace('<gap> <gap>', '<big_gap>')
            text = text.replace('<big_gap> <big_gap>', '<big_gap>')
            
            # Remove annotations
            text = self.patterns['annotations'].sub('', text)
            
            # Protect gaps, remove forbidden chars, restore gaps
            text = text.replace('<gap>', '\x00GAP\x00')
            text = text.replace('<big_gap>', '\x00BIG\x00')
            text = text.translate(self.forbidden_trans)
            text = text.replace('\x00GAP\x00', ' <gap> ')
            text = text.replace('\x00BIG\x00', ' <big_gap> ')
            
            # Normalize fractions
            text = self._normalize_fractions(text)
            
            # Remove repeated phrases
            text = self._remove_repeated_phrases(text)
        
        # Proper noun corrections
        if self.proper_noun_dict:
            text = self._fix_proper_nouns(text)
        
        # Clean up quotes
        text = self.patterns['quotes'].sub('"', text)
        
        # Punctuation cleanup
        text = self.patterns['punct_space'].sub(r'\1', text)
        text = self.patterns['repeated_punct'].sub(r'\1', text)
        
        # Final whitespace normalization
        text = self.patterns['whitespace'].sub(' ', text)
        text = text.strip().strip('-').strip()
        
        # Ensure proper sentence ending
        if text and text[-1] not in '.!?':
            text += '.'
        
        # Capitalize first letter
        if text:
            text = text[0].upper() + text[1:]
        
        return text
    
    def postprocess_batch(self, texts: List[str]) -> List[str]:
        """Process a batch of translations."""
        return [self.postprocess(t) for t in texts]

print("Postprocessor class defined")

## 3. Load Proper Noun Dictionary

In [ ]:
# Try to load proper noun dictionary
proper_noun_dict = {}

dict_paths = [
    config.proper_noun_dict_path,
    '/kaggle/working/proper_noun_dict.json',
    '/kaggle/input/akkadian-prepared-data/proper_noun_dict.json',
    'proper_noun_dict.json'
]

for path in dict_paths:
    if Path(path).exists():
        with open(path, 'r', encoding='utf-8') as f:
            proper_noun_dict = json.load(f)
        print(f"Loaded {len(proper_noun_dict)} proper noun entries from {path}")
        break

if not proper_noun_dict:
    print("Warning: No proper noun dictionary found. Proceeding without proper noun corrections.")

# Initialize postprocessor
postprocessor = EnhancedPostprocessor(
    proper_noun_dict=proper_noun_dict,
    aggressive=config.aggressive_postprocessing
)
print("Postprocessor initialized")

## 4. Load Model

In [ ]:
# Clear memory before loading model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loading model from {config.model_path}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_path)

# Load model with fp16 for memory efficiency
model = AutoModelForSeq2SeqLM.from_pretrained(
    config.model_path,
    torch_dtype=torch.float16 if config.use_mixed_precision else torch.float32,
    low_cpu_mem_usage=True
)
model = model.to(config.device)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {num_params:,} parameters")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Apply BetterTransformer if available
if config.use_better_transformer and torch.cuda.is_available():
    try:
        from optimum.bettertransformer import BetterTransformer
        model = BetterTransformer.transform(model)
        print("BetterTransformer applied")
    except ImportError:
        print("optimum not available, skipping BetterTransformer")
    except Exception as e:
        print(f"BetterTransformer failed: {e}")

## 5. Load Test Data

In [ ]:
# Find test data
test_paths = [
    config.test_data_path,
    '/kaggle/input/deep-past-initiative-machine-translation/test.csv',
    'data/deep-past-initiative-machine-translation/test.csv',
    'test.csv'
]

test_df = None
for path in test_paths:
    if Path(path).exists():
        test_df = pd.read_csv(path)
        print(f"Loaded {len(test_df)} test samples from {path}")
        break

if test_df is None:
    raise FileNotFoundError("Could not find test.csv")

print(f"\nTest data columns: {test_df.columns.tolist()}")
print(test_df.head())

## 6. Inference Dataset and DataLoader

In [ ]:
class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor, task_prefix: str):
        self.ids = df['id'].tolist()
        raw_texts = df['transliteration'].tolist()
        self.texts = [task_prefix + preprocessor.preprocess(t) for t in raw_texts]
        
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx]

# Create dataset
dataset = AkkadianDataset(test_df, preprocessor, config.task_prefix)
print(f"Created dataset with {len(dataset)} samples")

# Sample input
print(f"\nSample input: {dataset.texts[0][:100]}...")

In [ ]:
def collate_fn(batch):
    """Collate function for DataLoader."""
    ids = [item[0] for item in batch]
    texts = [item[1] for item in batch]
    
    tokenized = tokenizer(
        texts,
        max_length=config.max_input_length,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    
    return ids, tokenized

# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    collate_fn=collate_fn,
    pin_memory=True
)

print(f"Created DataLoader with {len(dataloader)} batches")

## 7. Length-Adaptive Generation Config

In [ ]:
def get_generation_config(input_length: int, config: InferenceConfig) -> dict:
    """
    Get generation config based on input length.
    Shorter inputs need less length penalty to avoid over-generation.
    """
    if not config.use_length_adaptive:
        return {
            'num_beams': config.num_beams,
            'max_new_tokens': config.max_new_tokens,
            'length_penalty': config.length_penalty,
            'early_stopping': config.early_stopping,
        }
    
    if input_length < 50:
        return {
            'num_beams': 6,
            'max_new_tokens': 128,
            'length_penalty': 1.2,
            'early_stopping': True,
        }
    elif input_length < 150:
        return {
            'num_beams': 8,
            'max_new_tokens': 256,
            'length_penalty': 1.5,
            'early_stopping': True,
        }
    else:
        return {
            'num_beams': 8,
            'max_new_tokens': 512,
            'length_penalty': 1.8,
            'early_stopping': True,
        }

print("Generation config function defined")

## 8. Run Inference

In [ ]:
def run_inference(model, dataloader, tokenizer, postprocessor, config):
    """
    Run inference on all test samples.
    """
    results = []
    
    model.eval()
    
    with torch.inference_mode():
        for batch_idx, (batch_ids, tokenized) in enumerate(tqdm(dataloader, desc="Translating")):
            try:
                input_ids = tokenized.input_ids.to(config.device)
                attention_mask = tokenized.attention_mask.to(config.device)
                
                # Get average input length for this batch
                avg_length = attention_mask.sum(dim=1).float().mean().item()
                gen_config = get_generation_config(int(avg_length), config)
                
                # Add no_repeat_ngram if specified
                if config.no_repeat_ngram_size > 0:
                    gen_config['no_repeat_ngram_size'] = config.no_repeat_ngram_size
                
                # Generate with mixed precision if enabled
                if config.use_mixed_precision:
                    with autocast():
                        outputs = model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            **gen_config
                        )
                else:
                    outputs = model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        **gen_config
                    )
                
                # Decode
                translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
                
                # Post-process
                cleaned = postprocessor.postprocess_batch(translations)
                
                # Store results
                for sample_id, translation in zip(batch_ids, cleaned):
                    results.append({'id': sample_id, 'translation': translation})
                
                # Periodic cache clearing
                if torch.cuda.is_available() and batch_idx % 20 == 0:
                    torch.cuda.empty_cache()
                    
            except Exception as e:
                print(f"Error in batch {batch_idx}: {e}")
                for sample_id in batch_ids:
                    results.append({'id': sample_id, 'translation': ''})
    
    return pd.DataFrame(results)

print("Inference function defined")

In [ ]:
# Run inference
print("\n" + "="*60)
print("STARTING INFERENCE")
print("="*60)

results_df = run_inference(model, dataloader, tokenizer, postprocessor, config)

print("\n" + "="*60)
print("INFERENCE COMPLETE")
print("="*60)

In [ ]:
# Sort by ID to ensure correct order
results_df = results_df.sort_values('id').reset_index(drop=True)

print(f"\nResults shape: {results_df.shape}")
print("\nSample translations:")
for i in range(min(3, len(results_df))):
    row = results_df.iloc[i]
    print(f"\n[ID {row['id']}]")
    print(f"Translation: {row['translation'][:150]}..." if len(str(row['translation'])) > 150 else f"Translation: {row['translation']}")

## 9. Generate Submission

In [ ]:
# Ensure submission has correct format
submission_df = results_df[['id', 'translation']].copy()

# Verify no empty translations
empty_count = (submission_df['translation'].isna() | (submission_df['translation'] == '')).sum()
if empty_count > 0:
    print(f"Warning: {empty_count} empty translations found")
    # Fill empty translations with placeholder
    submission_df['translation'] = submission_df['translation'].fillna('<gap>')
    submission_df.loc[submission_df['translation'] == '', 'translation'] = '<gap>'

# Save submission
submission_path = Path(config.output_dir) / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"\nSubmission saved to {submission_path}")
print(f"Submission shape: {submission_df.shape}")
print(f"File size: {submission_path.stat().st_size:,} bytes")

In [ ]:
# Verify submission format
print("\nSubmission preview:")
print(submission_df.head())

## 10. Summary

In [ ]:
print("="*60)
print("INFERENCE SUMMARY")
print("="*60)
print(f"\nModel: {config.model_path}")
print(f"Test samples: {len(test_df)}")
print(f"Translations generated: {len(submission_df)}")
print(f"\nGeneration parameters:")
print(f"  Beams: {config.num_beams}")
print(f"  Length penalty: {config.length_penalty}")
print(f"  Length adaptive: {config.use_length_adaptive}")
print(f"\nPost-processing:")
print(f"  Aggressive: {config.aggressive_postprocessing}")
print(f"  Proper noun correction: {config.use_proper_noun_correction}")
print(f"  Proper noun entries: {len(proper_noun_dict)}")
print(f"\nSubmission: {submission_path}")
print("="*60)

In [ ]:
# Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
print("\nDone!")